# Semenov et al. (2025), Figure S1, from dandiset 001272 (data elsewhere)

Paper. Semenov I, Kim V, Silkuniene G, Pakhomov AG (2025), *Cell Reports Physical Science* 6,
102660, [doi:10.1016/j.xcrp.2025.102660](https://doi.org/10.1016/j.xcrp.2025.102660); the claims
file was written from the NIH author manuscript (PMC12320893) with its supplement.

Dataset. The claim figs1-di8-anepps-control has `data: elsewhere`, in
[DANDI:001272](https://dandiarchive.org/dandiset/001272/0.250805.1358), version 0.250805.1358
("The Use of Di8-ANEPPS to Evaluate Neuronal Sensitivity to Unmodulated Sine Waves"), the most
recent published version; the paper lists the dandiset without a version.

| Claim | Tested here |
|---|---|
| figs1-di8-anepps-control | yes |

No random numbers are used.

In [ ]:
import time
from collections import Counter

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats

import fastimg

pd.set_option("display.width", 200)
started = time.time()
sessions = fastimg.read_all(fastimg.list_assets(fastimg.S1_DANDISET, fastimg.S1_VERSION))
print(f"{len(sessions)} sessions read from dandiset {fastimg.S1_DANDISET} version {fastimg.S1_VERSION} "
      f"in {time.time() - started:.0f} s")

## Data check

In [ ]:
info = pd.DataFrame([{k: s[k] for k in ("subject_id", "amplitude_v", "species", "rate_hz", "shape")} for s in sessions])
print("Sessions per subject id:", dict(Counter(info.subject_id)))
print("Species:", dict(Counter(info.species)), "; rates:", dict(Counter(info.rate_hz)), "; shapes:", dict(Counter(info["shape"])))
print("Amplitudes (V) per subject id:")
print(info.groupby("subject_id").amplitude_v.agg(["min", "max", "count"]).to_string())
print("Descriptions naming Di-8-ANEPPS:", sum("Di-8-ANEPPS" in s["description"] for s in sessions), "of", len(sessions))
print("Descriptions naming tetrodotoxin as present:", sum("presence of" in s["description"] for s in sessions))
print("\nDescription of one session:\n", sessions[0]["description"])

The dandiset has three neurons (subject ids, all recorded on 2024-08-06) where the paper reports
eight. Each session is 1,200 images of 16 x 16 pixels, about 389 ms; the stimulus timing is not in
the files and is taken from the Methods (50 ms before the stimulus, 250 ms during).

The traces are inverted, as Di-8-ANEPPS emission falls with depolarization (Methods). The check
below compares the mean inverted dF before the stated onset, during the stimulus and after it,
over all sessions: a response confined to the stated window supports the timing.

In [ ]:
fs = info.rate_hz.iloc[0]
on, off = int(fastimg.S1_STIM_START_S * fs), int((fastimg.S1_STIM_START_S + fastimg.S1_STIM_S) * fs)
for s in sessions:
    s["dF"] = -fastimg.delta_f(s["frame_means"], s["rate_hz"], fastimg.S1_STIM_START_S)
    s["sd_pre"] = float(np.nanstd(s["dF"][fastimg.DROP_FRAMES:on]))
x = np.array([s["dF"] for s in sessions])
print(f"Mean inverted dF (%): before onset {np.nanmean(x[:, fastimg.DROP_FRAMES:on]):.2f}, "
      f"first 25 ms of the stimulus {np.nanmean(x[:, on:on + int(0.025 * fs)]):.2f}, "
      f"whole stimulus {np.nanmean(x[:, on:off]):.2f}, last 50 ms after it {np.nanmean(x[:, -int(0.05 * fs):]):.2f}")
step = np.nanmean(x[:, on:on + 10], axis=1) - np.nanmean(x[:, on - 10:on], axis=1)
print(f"Step at the stated onset (10 frames after minus 10 before), median over sessions: {np.median(step):.2f}%; "
      f"the same 50 frames earlier: {np.median(np.nanmean(x[:, on - 40:on - 30], 1) - np.nanmean(x[:, on - 50:on - 40], 1)):.2f}%")
print("Pre-stimulus SD of dF (%), range over sessions:", round(min(s["sd_pre"] for s in sessions), 2), "to",
      round(max(s["sd_pre"] for s in sessions), 2))

The inverted dF rises from the stated onset (the median step at the onset is larger than the same
step 50 frames earlier) but stays up after the stimulus, so part of the rise is the bleaching of
the dye, which the inversion turns into an apparent depolarization. The stated timing is consistent
with the data, not shown by them.

## figs1-di8-anepps-control

> With Di-8-ANEPPS instead of FluoVolt, in eight neurons, near-threshold stimulation elicited one or
> a few delayed APs and stronger fields induced sustained firing ending in sustained
> depolarization, as with FluoVolt.

Criterion: traces inverted; AP detection readings (a) a peak above 5% dF, (b) a rise of at least
5% within 5 ms, (c) a peak above 5 SD of the pre-stimulus trace. Reproduced when, under every
reading, every neuron fires, first-AP latency falls with field in more than half, and the highest
field ends in depolarization as in fig3c-sustained-firing-2s in more than half. Taken here (standard
choices, as the criterion does not fix them): latency falls when the Spearman correlation of field
and first-AP latency over a neuron's sessions with APs is negative; the highest field ends in
depolarization when its sessions have no APs in the last 100 ms of the stimulus and mean dF above
5% there (the window of the claims file's block test for 250-ms stimuli, fig2-modulated-locks-to-beats).
AP detection is the helper's: a 1-ms running mean, and a peak of reading (a) or (c) must stand
above its surroundings by the detection level. The first run's detector is a variant.

In [ ]:
rows = []
last = off - int(0.1 * fs)
for s in sessions:
    row = {"neuron": s["subject_id"], "amplitude_v": s["amplitude_v"], "sd_pre": round(s["sd_pre"], 2),
           "mean_dF_last100": float(np.nanmean(s["dF"][last:off]))}
    for r in "abc":
        p = fastimg.detect(s["dF"], r, s["rate_hz"], level=5 * s["sd_pre"])
        p = p[(p >= on) & (p < off)]
        row[f"{r}_aps"] = len(p)
        row[f"{r}_latency_ms"] = 1000 * (p[0] - on) / fs if len(p) else np.nan
        row[f"{r}_last100"] = int((p >= last).sum())
    rows.append(row)
table = pd.DataFrame(rows).sort_values(["neuron", "amplitude_v"])
print(table.round(2).to_string(index=False))

In [ ]:
rows = []
for r in "abc":
    for neuron, sub in table.groupby("neuron"):
        fired = sub[sub[f"{r}_aps"] > 0]
        rho = stats.spearmanr(fired.amplitude_v, fired[f"{r}_latency_ms"]).correlation if fired.amplitude_v.nunique() > 1 else np.nan
        top = sub[sub.amplitude_v == sub.amplitude_v.max()]
        rows.append({"reading": r, "neuron": neuron, "sessions_with_aps": len(fired), "of": len(sub),
                     "fires": len(fired) > 0, "latency_rho": rho, "latency_falls": bool(rho < 0),
                     "highest_field_v": top.amplitude_v.iloc[0], "highest_aps_last100": int(top[f"{r}_last100"].sum()),
                     "highest_mean_dF_last100": top.mean_dF_last100.mean(),
                     "ends_depolarized": bool((top[f"{r}_last100"] == 0).all() and top.mean_dF_last100.mean() > 5)})
per = pd.DataFrame(rows)
print(per.round(2).to_string(index=False))
summary = per.groupby("reading").agg(all_fire=("fires", "all"), latency_falls=("latency_falls", "sum"),
                                     ends_depolarized=("ends_depolarized", "sum"), neurons=("neuron", "count"))
summary["holds"] = summary.all_fire & (summary.latency_falls > summary.neurons / 2) & (summary.ends_depolarized > summary.neurons / 2)
print("\n", summary.to_string())
print("\nLargest mean inverted dF over the last 100 ms of any session:", round(table.mean_dF_last100.max(), 2), "%")

# Variants. (1) The first run's detector (no smoothing, prominence 2.5% for readings a and c). (2) Reading (c)
# without APs in the first 5 ms of the stimulus, which may be an onset artefact. (3) The level of
# "ends in depolarization" taken as 5 SD of the neuron's pre-stimulus trace in place of the FluoVolt 5%
# (the claims file's 5% is a FluoVolt level; see the question to a reader in the attempt record).
def per_neuron(tab, level_col=None):
    rows = []
    for r in "abc":
        for neuron, sub in tab.groupby("neuron"):
            fired = sub[sub[f"{r}_aps"] > 0]
            rho = stats.spearmanr(fired.amplitude_v, fired[f"{r}_latency_ms"]).correlation if fired.amplitude_v.nunique() > 1 else np.nan
            top = sub[sub.amplitude_v == sub.amplitude_v.max()]
            level = 5 if level_col is None else 5 * top[level_col].mean()
            rows.append({"reading": r, "neuron": neuron, "fires": len(fired) > 0, "latency_falls": bool(rho < 0),
                         "level": level, "ends_depolarized": bool((top[f"{r}_last100"] == 0).all() and top.mean_dF_last100.mean() > level)})
    out = pd.DataFrame(rows)
    s = out.groupby("reading").agg(all_fire=("fires", "all"), latency_falls=("latency_falls", "sum"),
                                   ends_depolarized=("ends_depolarized", "sum"), neurons=("neuron", "count"))
    s["holds"] = s.all_fire & (s.latency_falls > s.neurons / 2) & (s.ends_depolarized > s.neurons / 2)
    return out, s
variant_rows = []
for s in sessions:
    row = {"neuron": s["subject_id"], "amplitude_v": s["amplitude_v"], "sd_pre": s["sd_pre"],
           "mean_dF_last100": float(np.nanmean(s["dF"][last:off]))}
    for r in "abc":
        p = fastimg.detect(s["dF"], r, s["rate_hz"], level=5 * s["sd_pre"], smooth=1,
                           prominence=fastimg.FIRST_RUN_PROMINENCE if r in "ac" else None)
        p = p[(p >= on) & (p < off)]
        row[f"{r}_aps"] = len(p); row[f"{r}_latency_ms"] = 1000 * (p[0] - on) / fs if len(p) else np.nan
        row[f"{r}_last100"] = int((p >= last).sum())
    variant_rows.append(row)
first_run = pd.DataFrame(variant_rows)
late_c = table.copy()
early = late_c.c_latency_ms < 5
print("\nSessions whose first AP under reading (c) comes within 5 ms of the onset:", int(early.sum()), "of",
      int(late_c.c_aps.gt(0).sum()), "sessions with APs")
late_c.loc[early, ["c_aps", "c_latency_ms"]] = [0, np.nan]
for name, tab, lev in [("first run's detector", first_run, None), ("reading (c) without the first 5 ms", late_c, None),
                       ("level 5 SD of the pre-stimulus trace", table, "sd_pre")]:
    out, s = per_neuron(tab, lev)
    print(f"\nVariant: {name}")
    print(s.to_string())
    if lev:
        print("Levels (%):", out[out.reading == "a"].set_index("neuron").level.round(2).to_dict())

In [ ]:
# Every neuron at its highest field (all sessions at that field), with the mean of its lowest-field
# sessions as the bleaching drift, the APs of reading (c), and the first 5 ms after the onset shaded.
fig, axes = plt.subplots(3, 1, figsize=(10, 7), sharex=True)
t = (np.arange(len(sessions[0]["dF"])) - on) / fs * 1000
lo, hi = np.inf, -np.inf
for ax, neuron in zip(axes, sorted(info.subject_id.unique())):
    sub = [s for s in sessions if s["subject_id"] == neuron]
    top_v = max(s["amplitude_v"] for s in sub)
    low_v = min(s["amplitude_v"] for s in sub)
    drift = np.nanmean([s["dF"] for s in sub if s["amplitude_v"] == low_v], axis=0)
    ax.plot(t, drift, c="grey", lw=1, label=f"mean at the lowest field, {low_v} V")
    for s in (s for s in sub if s["amplitude_v"] == top_v):
        ax.plot(t, s["dF"], lw=0.5, label=f"{top_v} V")
        p = fastimg.detect(s["dF"], "c", s["rate_hz"], level=5 * s["sd_pre"])
        p = p[(p >= on) & (p < off)]
        ax.plot(t[p], s["dF"][p], "k|", ms=8)
        lo, hi = min(lo, np.nanmin(s["dF"])), max(hi, np.nanmax(s["dF"]))
    ax.axvspan(0, 5, color="orange", alpha=0.3)
    ax.axvline(0, c="k", ls=":"); ax.axvline(250, c="k", ls=":"); ax.axhline(5, c="r", ls="--", lw=0.6)
    ax.set_ylim(-4, 8); ax.set_ylabel(neuron, fontsize=8); ax.legend(fontsize=6, ncol=4)
axes[-1].set_xlabel("time from the stated stimulus onset (ms); red dashed: 5% dF; orange: first 5 ms")
plt.show()
print(f"Inverted dF of the plotted highest-field sessions ranges {lo:.1f} to {hi:.1f}% (axes -4 to 8%)")
for neuron, sub in table.groupby("neuron"):
    lowest, highest = sub[sub.amplitude_v == sub.amplitude_v.min()], sub[sub.amplitude_v == sub.amplitude_v.max()]
    print(f"{neuron}: mean inverted dF over the last 100 ms, lowest field {lowest.mean_dF_last100.mean():.2f}%, "
          f"highest field {highest.mean_dF_last100.mean():.2f}%")

**Result: not reproduced.** The criterion holds under none of the three readings (`holds` is
False for a, b and c). In no neuron does the highest field end in depolarization by the
criterion's 5% level; the largest mean inverted dF over the last 100 ms of any session is 3.2%.
Under reading (c), 5 SD of the pre-stimulus trace, every neuron fires and first-AP latency falls
with field in all 3; under readings (a) and (b) the 5% level finds APs only in neuron 3_20240806
(4 sessions under a, none under b). In 19 of 52 sessions with APs under reading (c) the first AP
comes within 5 ms of the onset, which may be an onset artefact; without those, latency falls in 2
of 3. The variants do not change the outcome: with the first run's detector, without the first
5 ms, or with the depolarization level set to 5 SD of each neuron's pre-stimulus trace (2.2 to
2.5%), no reading has the highest field end in depolarization in more than one neuron. The inverted
dF over the last 100 ms at the highest field (0.92%, 3.20% and 1.58%) is close to that at the
lowest field (1.53%, 1.34% and 1.27%), part of which is bleaching. The Di-8-ANEPPS signals are
smaller than the FluoVolt ones the 5% level comes from, and the dataset has three neurons where
the paper has eight; whether the 5% level fits this dye is asked of a reader (attempt record).

## Summary

| Claim | Outcome |
|---|---|
| figs1-di8-anepps-control | not reproduced |

A failure to reproduce here may be a failure of this attempt, not of the paper: see the
reasoning and the differences from the paper's method.

How the analysis differs from the paper's: F is the whole 16 x 16 frame; the stimulus timing is the
Methods' (not in the files); the readings of "latency falls" and "ends in depolarization" are
standard choices stated above; AP detection on a 1-ms running mean (standard choice).

What was found about the dataset: three neurons where the paper reports eight; no stimulus timing
in the files.